# NITK-UsoundSim: team-by-team demo

**A 2-D ultrasound B-mode simulator, built as 7 team modules.** A virtual probe sends a sound pulse into a virtual tissue (the *phantom*), records the echoes, and turns them into the grey ultrasound image a scanner shows.

Probe: **Philips L12-4 (FUS4103) reference model**: 128 elements, 0.30 mm pitch, 8 MHz centre (4-12 MHz range). Philips publishes the 128 elements and the 4-12 MHz range; the pitch and the centre frequency are T2's documented assumptions.

The simulator runs **inside this notebook**, one section per team, in pipeline order. Every section has the same four parts:

| Part | What it tells you |
|---|---|
| **Receives** | the data that arrives from the previous team |
| **Does** | the physics / algorithm, with the equation |
| **Output** | the data handed to the next team (name, shape, units) |
| **How to read the plots** | what you should see, and why it proves the step worked |

Nothing is loaded from `results/`: every picture below is computed live.

## The pipeline at a glance

```
 T1 Transducer ──► T2 Transmit ──► T3 Propagation ──► T4 Tissue + RF ──► T5 Receive BF ──► T6 B-mode ──► T7 Recon + Post
 128 element      pulse, delays    sound travels       echoes from        delay-and-sum     envelope +     scan conversion
 positions        (when each       to each point,      every scatterer    → one line per    log            + despeckle filter
                  element fires)   and back            summed per element  x-position        compression    → FINAL IMAGE
```

| Team | Main output handed to the next team | Code |
|---|---|---|
| T1 Transducer | `element_positions` (128,) m | `simulator/config.py` → T3's `create_linear_array` |
| T2 Transmit Beamforming | `pulse`, `tx_delays` (128,) s, `apodization`, `transmit_field` | `teams/T2_TX_Beamforming/Usound/src/beamformer.py` |
| T3 Acoustic Propagation | `propagated_wavefield` (incident wave), `travel_times` (128,) s, echoes at the elements | `teams/T3_Acoustic_Propagation/usound_sim/nitk_usoundsim/acoustic_propagation.py` |
| T4 Tissue Interaction + RF | phantom, reflected wave, **`raw_rf`** (128, 3574) + `t_axis` | `teams/T4_Tissue_Interaction/` (`phantom.py`, `tissue_interaction.py`, `tissue_phantoms.py`) |
| T5 Receive Beamforming | **`beamformed_rf`** (3118, 256) + `z_axis` | `teams/T5_RX_Beamforming/receive_beamforming.py` |
| T6 B-mode Formation | envelope, log compression, **`bmode`** (3118, 256) in [0, 1] | `teams/T6_B_mode/bmode_formation.py` |
| T7 Image Reconstruction & Post-Processing | scan-converted `image` (512, 256), **`final_bmode`** | `teams/T7_Image_Recon_Post_Processing/post_processing.py` |

All shared numbers (speed of sound, sampling rate, probe, phantom size, ...) live in one file, `simulator/config.py`, so no team hard-codes a different value.

## Words used in this notebook

| Term | Meaning |
|---|---|
| **Element** | one of the 128 small piezo crystals in the probe; each one sends and receives sound |
| **Pitch** | centre-to-centre spacing of the elements (0.30 mm) |
| **λ (wavelength)** | c / f0 = 1540 / 8 MHz = 0.1925 mm; sets the finest detail the image can show |
| **Scatterer** | a tiny point in the tissue that reflects a little sound (cells, fibres) |
| **Phantom** | the virtual tissue: a list of scatterers with positions and strengths |
| **RF data** | the raw echo voltage on each element vs time, before any image is formed |
| **Beamforming** | lining up the echoes from all elements in time and adding them, to focus on one point |
| **Envelope** | the smooth outline (amplitude) of the oscillating RF signal |
| **Dynamic range (DR)** | how many dB of brightness the display shows (50 dB here) |
| **Speckle** | the grainy texture of ultrasound images, from many scatterers interfering |
| **Cyst** | a fluid-filled region with no scatterers: it appears black |

## How to run
- **Kernel:** **Python 3.12.4 / "Python 3 (ipykernel)"** (top-right), then **Run All**.
- **Time:** the whole notebook took **777 s (≈13 min)** on an 8-core Mac: each speckle phantom takes ≈4 min (96,000 scatterers × 128 elements). On Colab (2 CPUs) it is slower.
- **`KIND`** in the setup cell picks the phantom: `"cyst_lesion"` (default), `"scatterers"` or `"point_targets"` (fast, ≈3 s).

## Setup: shared `config.py` (single source of truth)

**What the next cell does**
1. Finds the project folder (the one that contains `simulator/config.py`), from whatever folder the notebook is opened in.
2. Imports `config`, which puts every team's folder on the Python path.
3. Imports each team's function that is used below:

| Import | Team | Role |
|---|---|---|
| `generate_pulse`, `forward_propagation` | T3 | pulse shape, sound travel |
| `calculate_tx_delays`, `make_apodization` | T2 | focused delays, window |
| `build_phantom`, `simulate_rf`, `incident_wave`, `point_echo` | T4 | phantom and raw RF |
| `tissue_interaction` | T4 | reflection at one scatterer |
| `das_beamform` | T5 | receive beamforming |
| `bmode_formation` | T6 | envelope + log compression |
| `scan_convert`, `postprocess`, `speckle_index` | T7 | image and filter |

4. Prints the main physical settings.

**Expected printout:** 128 elements, pitch 0.30 mm, f0 8 MHz, fs 40 MHz, c 1540 m/s, α 0.5 dB/MHz/cm, DR 50 dB; image x −10..+10 mm (256 lines), depth 0-40 mm.

In [ ]:
import sys, time
from pathlib import Path

# Find the project root (the folder that contains simulator/config.py), whatever folder the notebook runs from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulator" / "config.py").exists())
# A kernel started before the folders were reorganised may still hold modules from the old locations: drop them.
for _name, _mod in list(sys.modules.items()):
    _f = getattr(_mod, "__file__", None) or ""
    if "Medical Ultrasound" in _f and (not Path(_f).exists() or not _f.startswith(str(ROOT))):
        del sys.modules[_name]
sys.path = [p for p in sys.path if "Medical Ultrasound" not in p or Path(p).exists()]
print("Python:", sys.executable, sys.version.split()[0], "| project:", ROOT)
import cv2, numpy as np, matplotlib.pyplot as plt
from scipy.signal import hilbert

sys.path.insert(0, str(ROOT / "simulator"))      # project modules live in simulator/
import config
from tissue_phantoms import build_phantom, simulate_rf
from receive_beamforming import das_beamform
from bmode_formation import bmode_formation
from post_processing import scan_convert, postprocess, speckle_index, to_uint8
from nitk_usoundsim.acoustic_propagation import generate_pulse, forward_propagation
from tissue_phantoms import incident_wave, point_echo
from tissue_interaction import tissue_interaction
sys.path.insert(0, str(config.T2_DIR))           # teams/T2_TX_Beamforming/Usound
from src.beamformer import calculate_tx_delays, make_apodization     # T2's code

KIND = "cyst_lesion"        # or "scatterers" / "point_targets"
plt.rcParams["figure.dpi"] = 90
print(config.PROBE_NAME)
print(f"{config.N_ELEMENTS} elements | pitch {config.PITCH*1e3:.2f} mm | f0 {config.F0/1e6:g} MHz | fs {config.FS/1e6:g} MHz | c {config.C:g} m/s | "
      f"alpha {config.ALPHA_0} dB/MHz/cm | DR {config.DYNAMIC_RANGE_DB:g} dB")
print(f"imaged x {config.SCANLINE_X_MIN*1e3:+g}..{config.SCANLINE_X_MAX*1e3:+g} mm ({config.N_SCANLINES} lines), "
      f"depth {config.Z_MIN*1e3:g}-{config.Z_MAX*1e3:g} mm")

---
# T1: Transducer

**Role in the pipeline:** defines the probe: *where* the 128 elements sit. Every later step (delays, travel times, beamforming) is computed from these positions.

**Receives:** `N_ELEMENTS = 128`, `PITCH = 0.30 mm` from `config.py`.

**Does:** places the elements in a straight line along x, centred on x = 0, at depth z = 0 (the skin surface):

$$x_n = \left(n - \tfrac{N-1}{2}\right)\cdot \text{pitch}, \qquad n = 0 \ldots 127$$

It uses T3's `create_linear_array`, because T1's own code (`teams/T1_Transducer/src/`) is a piezo-material registry (material properties), not an array-geometry function.

**Output:** `element_positions`, shape (128,), metres. x = lateral, all at z = 0, element 0 at the most negative x.

**Why pitch matters:** the pitch here is 0.30 mm = **1.56 λ** (a textbook array uses ≤ λ/2). A coarse pitch can create *grating lobes* (false copies of a bright point). This is why the element directivity (T3/T4) and the F-number receive aperture (T5) were added, see those sections.

In [ ]:
element_positions = config.get_element_positions()
EX = element_positions
print("element_positions:", EX.shape, "| x from", round(EX[0]*1e3, 3), "to", round(EX[-1]*1e3, 3), "mm")
print(f"pitch {config.PITCH*1e3:.3f} mm = {config.PITCH/config.LAMBDA:.2f} lambda | aperture {config.APERTURE*1e3:.1f} mm")
plt.figure(figsize=(9, 1.6)); plt.plot(EX*1e3, np.zeros_like(EX), "s", ms=4)
plt.yticks([]); plt.xlabel("x (mm)"); plt.title(f"T1 output: {config.N_ELEMENTS} element positions"); plt.show()

**How to read the plot:** 128 evenly spaced squares from **−19.05 mm to +19.05 mm**, i.e. a **38.4 mm** wide probe face. The image is only formed for x = −10..+10 mm, well inside the probe, so every image line has elements on both sides of it.

---
# T2: Transmit Beamforming

**Role in the pipeline:** decides *what* each element sends and *when*, i.e. the shape of the sound beam going into the body.

**Receives:** `element_positions` (from T1).

**Does:**
1. **Pulse:** a short Gaussian-windowed sine burst at f0 = 8 MHz (T3's `generate_pulse`), about 3 cycles long. A short pulse gives good depth (axial) resolution.
2. **Transmit delays `tx_delays`:** the time each element waits before firing. For a focus at point (x_f, z_f), elements far from the focus fire first so that all wavefronts arrive together:
   $$\tau_n = \frac{\max_m d_m - d_n}{c}, \qquad d_n = \sqrt{(x_n - x_f)^2 + z_f^2}$$
3. **Apodization:** a weight per element (e.g. a Hann window) that tapers the edges of the aperture to reduce side lobes.

**Outputs:** `pulse` (at 8 MHz), `tx_delays` (128,) s, `apodization` (128,), `transmit_field` (beam map).

**What the imaging chain actually uses:** a **broadcast** transmit, i.e. all 128 elements fire at the same time (all delays 0) with no apodization. One broadcast shot lights up the whole field of view, so one simulation gives the whole image; T3 also has no per-element weight input. T2's **focused** delays (`calculate_tx_delays`, focus at 20 mm) are computed and shown for comparison. `tests/parameter_tests.py` measured that T2's focus makes the field **+18.36 dB** stronger at the focus.

**What the next cell does:** makes the pulse, the broadcast delays, T2's focused delays and T2's Hann window, and plots them.

In [ ]:
tp, pulse = generate_pulse(f0=config.F0)
tx_delays = config.get_tx_delays()                              # used: broadcast
tx_delays_focus = calculate_tx_delays(EX, 0.0, 0.020, config.C)  # T2, focus (0, 20) mm
apodization = make_apodization(config.N_ELEMENTS, "hann")       # T2's window (not applied in the chain)
print("pulse:", pulse.shape, "| tx_delays:", tx_delays.shape, "| T2 focused delays max", round(tx_delays_focus.max()*1e6, 3), "us")

fig, ax = plt.subplots(1, 3, figsize=(15, 3))
ax[0].plot(tp*1e6, pulse); ax[0].set(title=f"pulse ({config.F0/1e6:g} MHz)", xlabel="time (us)")
ax[1].plot(EX*1e3, tx_delays*1e6, label="broadcast (used)"); ax[1].plot(EX*1e3, tx_delays_focus*1e6, label="T2 focus 20 mm")
ax[1].set(title="tx_delays", xlabel="element x (mm)", ylabel="us"); ax[1].legend()
ax[2].plot(EX*1e3, np.ones_like(EX), label="broadcast (used)"); ax[2].plot(EX*1e3, apodization, label="T2 Hann")
ax[2].set(title="apodization", xlabel="element x (mm)"); ax[2].legend(); plt.tight_layout(); plt.show()

**How to read the plots:**
- **pulse:** a few oscillations at 8 MHz (period 0.125 µs) under a bell-shaped envelope.
- **tx_delays:** broadcast is a flat line at 0. T2's focused delays form a curve: the middle elements wait longest (max ≈ 4.95 µs) so the outer ones, which are farther from the focus, catch up.
- **apodization:** broadcast = 1 everywhere; T2's Hann window goes from 0 at the edges to 1 in the centre.

**What the next cell does:** computes the **transmit field**, the peak strength of the transmitted wave at every point of a 49 × 39 grid (x −12..+12 mm, z 2-40 mm), once with broadcast and once with T2's focus. Each point uses T3's forward propagation × element directivity (≈19 s).

In [ ]:
# transmit_field: peak of the incident wave over a grid (T3 forward_propagation x element directivity, each set of delays)
t_axis = config.get_t_axis()
gx, gz = np.linspace(-0.012, 0.012, 49), np.linspace(0.002, 0.040, 39)
def transmit_field(delays):
    f = np.zeros((gz.size, gx.size))
    for i, z in enumerate(gz):
        for j, x in enumerate(gx):
            inc = incident_wave({"x": x, "z": z}, t_axis, tau_tx=delays)
            f[i, j] = np.abs(hilbert(inc)).max()
    return f
t0 = time.time(); field_b, field_f = transmit_field(tx_delays), transmit_field(tx_delays_focus)
print(f"transmit_field: {field_b.shape} grid points, {time.time()-t0:.1f} s")
fig, ax = plt.subplots(1, 2, figsize=(9, 5))
for a, f, t in zip(ax, (field_b, field_f), ("broadcast (used)", "T2 focus at 20 mm")):
    im = a.imshow(20*np.log10(f/f.max()), cmap="inferno", vmin=-30, vmax=0, aspect="auto",
                  extent=[gx[0]*1e3, gx[-1]*1e3, gz[-1]*1e3, gz[0]*1e3])
    a.set(title=f"transmit_field: {t}", xlabel="x (mm)", ylabel="z (mm)")
fig.colorbar(im, ax=ax, label="dB re own max"); plt.show()

**How to read the transmit-field maps (dB, 0 dB = strongest point of each map):**
- **Broadcast (used):** a wide, fairly uniform beam covering the whole imaged width. That is what we want: every scatterer gets insonified by the same shot.
- **T2 focus at 20 mm:** the energy converges into a narrow bright waist at z ≈ 20 mm. Better lateral resolution at 20 mm, but only one line would be sharp per shot, so a full image would need many shots.

---
# T3: Acoustic Propagation

**Role in the pipeline:** the physics of sound travelling through tissue: how long it takes, and how much it weakens.

**Receives:** `element_positions`, `tx_delays`, `pulse`, and from config the speed of sound c = 1540 m/s and the attenuation α = 0.5 dB/MHz/cm.

**Does:**
1. **Forward propagation** (probe → point): for each element, the distance to the point is d_n = √((x − x_n)² + z²), so the wave arrives after **t_n = τ_n + d_n / c**.
2. **Attenuation:** tissue absorbs sound. The loss grows with frequency and distance:
   $$A(d) = 10^{-\alpha f_0 d / 20}$$
   At 8 MHz that is 4 dB per cm, one way. `tests/parameter_tests.py` measured −16.11 dB and −32.21 dB against the predicted −16 and −32 dB.
3. **Element directivity:** a real element of width w sends and hears best straight ahead and less at steep angles:
   $$D(\theta) = \mathrm{sinc}\!\left(\frac{w \sin\theta}{\lambda}\right)$$
   T3 treats elements as points, so this factor is applied on top of T3's output (T3's code is not changed). Without it, the coarse 1.56 λ pitch produced ghost copies of each point target only −11 to −13 dB below it. With it they are −21 to −34 dB.
4. **Return propagation** (point → every element): the echo travels back, with its own delay and attenuation, to each of the 128 elements.

**Outputs:** `propagated_wavefield` (the incident wave at the point), `travel_times` (128,) s, and the echo received on the 128 elements.

**What the next cell does:** shows all of this for **one test point at (0, 20) mm**.

In [ ]:
test_point = {"x": 0.0, "z": 0.020, "amp": 1.0}
_, d_tx, travel_times = forward_propagation(test_point, EX, t_axis, f0=config.F0, tau_tx=tx_delays)  # T3's travel times
propagated_wavefield = incident_wave(test_point, t_axis, tau_tx=tx_delays)   # with element directivity
echo_at_elements = point_echo(test_point, t_axis)                            # T3 return x directivity
print("propagated_wavefield:", propagated_wavefield.shape, "| travel_times:", travel_times.shape,
      f"{travel_times.min()*1e6:.2f}-{travel_times.max()*1e6:.2f} us | echo_at_elements:", echo_at_elements.shape)
m = (t_axis > 11e-6) & (t_axis < 20e-6)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(t_axis[m]*1e6, propagated_wavefield[m]); ax[0].set(title="incident wave at (0, 20) mm", xlabel="time (us)")
ax[1].plot(EX*1e3, travel_times*1e6, "."); ax[1].set(title="travel_times (one way)", xlabel="element x (mm)", ylabel="us")
e = (t_axis > 24e-6) & (t_axis < 33e-6); lim = np.abs(echo_at_elements).max()
ax[2].imshow(echo_at_elements[:, e].T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
             extent=[0.5, config.N_ELEMENTS + 0.5, t_axis[e][-1]*1e6, t_axis[e][0]*1e6])
ax[2].set(title=f"echo received on {config.N_ELEMENTS} elements", xlabel="element #", ylabel="time (us)"); plt.tight_layout(); plt.show()

**How to read the plots:**
- **Incident wave at (0, 20) mm:** the transmitted pulse arriving at the point, ≈13 µs after firing (20 mm / 1540 m/s = 13.0 µs).
- **travel_times:** a U-shaped curve from **12.99 µs** (centre element, straight above the point) to **17.94 µs** (edge elements, farthest away). This curvature is exactly what T5 must undo later.
- **Echo on 128 elements:** the returning echo forms a **hyperbola**: it reaches the centre elements first (≈26 µs round trip) and the edge elements later. The edges are fainter because of directivity (steep angle) and the longer path.

---
# T4: Tissue Interaction + Echo / RF Generation

**Role in the pipeline:** builds the virtual body and produces the **raw RF data**, the exact signals a real probe would record.

**Receives:** T3's propagation functions, and the phantom settings from config.

**Does:**
1. **Build the phantom** (`build_phantom`, using T4's `phantom.py`). Three phantoms are available:

| `KIND` | Contents | Tests |
|---|---|---|
| `point_targets` | 3 bright points at (−5, 10), (0, 20), (+5, 30) mm | resolution and position accuracy |
| `scatterers` | 96,000 random scatterers (10⁸ per m²) in x ±12 mm, z 0-40 mm | speckle texture |
| `cyst_lesion` | the same speckle + an **anechoic cyst**: circle, centre (0, 30) mm, radius 6 mm, no scatterers inside | contrast (a dark hole must look dark) |

2. **Tissue interaction** (`tissue_interaction`, T4's code): each scatterer re-radiates the incident wave scaled by its reflectivity (amplitude `amp`; related to the acoustic-impedance change, background 1.63 MRayl).
3. **Sum everything** (`simulate_rf`): for every scatterer, incident wave → reflection → return to all 128 elements, and all echoes are added (sound waves superpose linearly). With 96,000 scatterers this is the slowest step, so it runs in parallel on all CPU cores.

**Output:** `raw_rf`, shape **(128, 3574)** = elements × time samples, with `t_axis` (3574 samples at 40 MHz). `t_axis` starts at **−0.75 µs** (half a pulse before the pulse centre), not at 0. Every later delay lookup must use this real axis (`np.interp(tau, t_axis, rf)`), never `tau × fs`, otherwise the image shifts by ≈0.58 mm.

**What the next cell does:** builds the phantom chosen by `KIND`, and shows the reflection at its strongest scatterer.

In [ ]:
phantom = build_phantom(KIND)
sx = np.array([s["x"] for s in phantom])*1e3; sz = np.array([s["z"] for s in phantom])*1e3
amp = np.array([s["amp"] for s in phantom])
s0 = max(phantom, key=lambda s: abs(s["amp"]))
inc0 = incident_wave(s0, t_axis, tau_tx=tx_delays)
reflected = tissue_interaction(inc0, s0)
print(f"phantom {KIND}: {len(phantom)} scatterers | reflected wave example: amp {s0['amp']:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw=dict(width_ratios=[1, 2]))
ax[0].scatter(sx, sz, s=30 if len(phantom) < 10 else 0.2, c=np.abs(amp), cmap="viridis")
ax[0].invert_yaxis(); ax[0].set(title=f"phantom: {KIND}", xlabel="x (mm)", ylabel="z (mm)")
k = np.abs(inc0) > 1e-3 * np.abs(inc0).max(); k = slice(np.argmax(k) - 20, len(k) - np.argmax(k[::-1]) + 20)
ax[1].plot(t_axis[k]*1e6, inc0[k], label="incident"); ax[1].plot(t_axis[k]*1e6, reflected[k], "--", label="reflected")
ax[1].set(title="tissue_interaction at one scatterer", xlabel="time (us)"); ax[1].legend(); plt.tight_layout(); plt.show()

**How to read the plots:**
- **Left, the phantom:** each dot is a scatterer (colour = strength). For `cyst_lesion` you see a dense cloud with an **empty circle** at 30 mm depth: the cyst.
- **Right, one scatterer:** the reflected wave is the incident wave multiplied by the scatterer's reflectivity: same shape, different amplitude (a negative amplitude flips it).

**What the next cell does:** runs the full RF simulation, all scatterers × 128 elements (≈4 min for the speckle phantoms; 231 s measured on the 8-core Mac).

In [ ]:
t0 = time.time()
raw_rf, t_axis = simulate_rf(phantom)
print(f"raw_rf: {raw_rf.shape} (elements x samples), t_axis from {t_axis[0]*1e6:.3f} us, {time.time()-t0:.1f} s")
lim = np.percentile(np.abs(raw_rf), 99.9)
plt.figure(figsize=(7, 5)); plt.imshow(raw_rf.T, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
                                      extent=[0.5, config.N_ELEMENTS + 0.5, t_axis[-1]*1e6, t_axis[0]*1e6])
plt.ylim(60, 0); plt.xlabel("element #"); plt.ylabel("time (us)"); plt.title(f"T4 output: {config.N_ELEMENTS}-channel raw_rf"); plt.show()

**How to read the plot (raw RF, elements × time):** this is the data a real scanner records before any processing. You cannot see the cyst or any structure yet: each column is one element, and every scatterer's echo is a faint hyperbola, all overlapping. Turning this into a picture is the job of T5 to T7. The signal also gets weaker with time (= depth), because of attenuation.

---
# T5: Receive Beamforming (Delay-and-Sum, DAS)

**Role in the pipeline:** turns the 128 raw channels into **image lines**, by focusing on every pixel.

**Receives:** `raw_rf`, `t_axis`, `element_positions`, and 256 scan-line positions (x = −10..+10 mm, 0.078 mm apart).

**Does:** for every scan line x and every depth z (dynamic receive focus):
1. **Round-trip delay** to each element: transmit (broadcast, straight down to depth z) + return from (x, z) to element n:
   $$\tau_n(x, z) = \frac{z}{c} + \frac{\sqrt{(x - x_n)^2 + z^2}}{c}$$
2. **Read** each channel at that time: `np.interp(tau, t_axis, rf)` on the **real** time axis.
3. **Weight and sum:** multiply by a Hann window and add the 128 samples. Echoes from (x, z) line up and add strongly; echoes from elsewhere do not line up and cancel.
4. **Dynamic aperture, F-number 1:** only elements within half-width z / (2·F#) of the line are used, so the aperture grows with depth (≈20 mm wide at z = 20 mm). This removed a grating-lobe streak that was −17 dB and is now below −100 dB. It widens the point width to 0.47 mm.

**Output:** `beamformed_rf`, shape **(3118, 256)** = depth samples × scan lines, with `z_axis` (0-60 mm, step 0.019 mm).

T5's own test script `teams/T5_RX_Beamforming/verify_beamformer.py` passes 4/4 (depth errors 0.008-0.040 mm).

In [ ]:
x_lines = config.get_scanline_positions()
beamformed_rf, z_axis = das_beamform(raw_rf, EX, x_lines, config, t_axis=t_axis)
print("beamformed_rf:", beamformed_rf.shape, "| z_axis", z_axis.shape, f"0-{z_axis[-1]*1e3:.0f} mm")
lim = np.percentile(np.abs(beamformed_rf), 99.9)
plt.figure(figsize=(4, 6)); plt.imshow(beamformed_rf, aspect="auto", cmap="gray", vmin=-lim, vmax=lim,
                                      extent=[x_lines[0]*1e3, x_lines[-1]*1e3, z_axis[-1]*1e3, z_axis[0]*1e3])
plt.ylim(40, 0); plt.xlabel("x (mm)"); plt.ylabel("z (mm)"); plt.title("T5 output: beamformed_rf"); plt.show()

**How to read the plot:** the first picture with recognisable structure. Each column is one scan line of RF (still oscillating, black/white stripes). For `cyst_lesion` the **dark circle at 30 mm** is already visible. The image is shown to 40 mm; the RF was formed to 60 mm only for T5's deep test point.

---
# T6: B-mode Formation (envelope + log compression)

**Role in the pipeline:** turns oscillating RF into **brightness** values.

**Receives:** `beamformed_rf` (3118, 256).

**Does** (`bmode_formation`), on every scan line along depth:
1. **Envelope detection** with the Hilbert transform, which gives the smooth amplitude of the RF:
   $$E(z, x) = \left| s(z, x) + j\,\mathcal{H}\{s\}(z, x) \right|$$
2. **Log compression:** echoes span a huge range (strong walls vs. weak tissue), so they are shown in decibels relative to the brightest point:
   $$L = 20 \log_{10}\!\left(E / \max E\right) \quad [\text{dB}, \le 0]$$
3. **Dynamic range 50 dB:** everything below −50 dB is black; the range is mapped to brightness 0..1:
   $$B = \frac{\mathrm{clip}(L, -50, 0) + 50}{50}$$

**Output:** `bmode`, shape (3118, 256), values 0 (≤ −50 dB) … 1 (0 dB). This is the raw B-mode array handed to T7.

**Why 50 dB?** It was chosen on evidence: the EPFL reference source uses 40 dB, and T6's earlier 60 dB had no source; 50 dB shows the speckle and the cyst clearly.

In [ ]:
envelope = np.abs(hilbert(beamformed_rf, axis=0))
log_env = 20*np.log10(np.maximum(envelope/envelope.max(), 1e-12))
bmode = bmode_formation(beamformed_rf)
print("envelope:", envelope.shape, "| bmode:", bmode.shape, f"range [{bmode.min():.2f}, {bmode.max():.2f}]")
li = np.argmin(np.abs(x_lines)); zm = (z_axis > 0.017) & (z_axis < 0.023)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1]))
ax[0].plot(z_axis[zm]*1e3, beamformed_rf[zm, li], lw=0.7, label="beamformed RF"); ax[0].plot(z_axis[zm]*1e3, envelope[zm, li], lw=2, label="envelope")
ax[0].set(title="envelope detection (centre line)", xlabel="depth (mm)"); ax[0].legend()
ax[1].plot(z_axis[zm]*1e3, log_env[zm, li]); ax[1].axhline(-config.DYNAMIC_RANGE_DB, ls="--", c="gray")
ax[1].set(title="log compression (dB)", xlabel="depth (mm)", ylim=(-80, 3))
ax[2].imshow(bmode, cmap="gray", vmin=0, vmax=1, aspect="auto"); ax[2].set(title="T6 output: raw bmode array", xlabel="scan line #", ylabel="depth sample #")
plt.tight_layout(); plt.show()

**How to read the plots:**
- **Left, envelope detection (centre line, 17-23 mm):** the thin line is the oscillating RF; the thick line is its envelope, which follows the peaks. Only the envelope is kept.
- **Middle, log compression:** the same envelope in dB. The dashed line is −50 dB: anything below it becomes black.
- **Right, raw bmode array:** the grey image in array coordinates (scan line # × depth sample #). It is stretched, because the axes are sample indices, not millimetres. T7 fixes that.

---
# T7: Image Reconstruction & Post-Processing

**Role in the pipeline:** turns the B-mode array into a correctly scaled **display image**, and cleans up the speckle.

**Receives:** `bmode` (3118, 256), scan-line positions, `z_axis`.

**Does:**
1. **Scan conversion** (`scan_convert`): a linear array has parallel, vertical scan lines, so the data are already a rectangular image. There is no fan / sector geometry. Scan conversion crops the depth to 0-40 mm and resamples (bilinear interpolation) onto **square pixels** of 0.078 mm. Square pixels matter because the filter's radius is in pixels.
2. **Despeckling** (`postprocess`): the **guided filter** (He et al.), chosen in the Post-Image-Processing notebook out of 10 filters (median, Lee, SRAD, NLM, wavelet, OBNLM, BM3D, guided, bilateral, TV). In every 9 × 9 window (r = 4) it fits `output = a · input + b` with
   $$a = \frac{\mathrm{var}}{\mathrm{var} + \varepsilon}$$
   - **Flat areas** (variance ≪ ε): a ≈ 0, so the output is the local mean and the speckle is smoothed.
   - **Edges** (variance ≫ ε): a ≈ 1, so the output ≈ the input and the edge is kept.

   ε = 0.001 is used.

**Outputs:** `image` (512, 256) at 0.078 mm/px (before the filter), and **`final_bmode`** (512, 256) in [0, 1], the final image.

**Speckle index** = mean of (local std / local mean); lower = smoother. It is printed below.

In [ ]:
image, x_img, z_img = scan_convert(bmode, x_lines, z_axis)
final_bmode = postprocess(image)
extent = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
print("image:", image.shape, f"pixel {(x_img[1]-x_img[0])*1e3:.3f} mm | final_bmode:", final_bmode.shape)
if KIND != "point_targets":
    print(f"speckle index: before {speckle_index(to_uint8(image)):.3f}, after {speckle_index(to_uint8(final_bmode)):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(7, 7))
for a, im, t in zip(ax, (image, final_bmode), ("scan-converted", "post-processed")):
    a.imshow(im, cmap="gray", vmin=0, vmax=1, extent=extent); a.set(title=t, xlabel="x (mm)", ylabel="z (mm)")
plt.tight_layout(); plt.show()

**How to read the plots:** left = after scan conversion, right = after the filter. Now the axes are real millimetres. For `cyst_lesion`: grey speckled tissue with a **black circle (the cyst) at 30 mm depth**, and the image gets darker with depth (attenuation; this simulator has no depth-gain / TGC correction). The filter's change is deliberately gentle (speckle index **0.308 → 0.241** in the verified run), so the two look similar at this size. The next section makes the difference visible.

### T7: seeing the filter's effect
The change is small on the full image, so it is shown four ways:
1. **Zoom** into the yellow box (cyst edge).
2. **Difference map ×10:** bright where the filter changed the pixel.
3. **Brightness along one row** (z = 20 mm): the filtered curve is smoother but follows the same shape.
4. **A stronger filter (ε 0.01) for comparison: not used.** It smooths much more (speckle index 0.308 → 0.129), but lowers edge sharpness to ≈0.78 of the original, and the notebook's rule requires ≥ 0.90.

This uses the cyst image (built here if `KIND` is not `"cyst_lesion"`, ≈4 min).

In [ ]:
if KIND == "cyst_lesion":
    image_c, x_img_c, z_img_c = image, x_img, z_img
else:
    t0 = time.time()
    rf_c, t_c = simulate_rf(build_phantom("cyst_lesion"))
    bf_c, z_c = das_beamform(rf_c, EX, x_lines, config, t_axis=t_c)
    image_c, x_img_c, z_img_c = scan_convert(bmode_formation(bf_c), x_lines, z_c)
    print(f"cyst image built in {time.time()-t0:.1f} s")
_keep = (image, x_img, z_img)
image, x_img, z_img = image_c, x_img_c, z_img_c
strong = postprocess(image, eps=0.01)            # stronger setting, NOT used (edges drop to ~0.78)
chosen = postprocess(image)                      # pipeline setting: r = 4, eps = 0.001
ext = [x_img[0]*1e3, x_img[-1]*1e3, z_img[-1]*1e3, z_img[0]*1e3]
zx = ((x_img*1e3 >= -9) & (x_img*1e3 <= 1)); zz = ((z_img*1e3 >= 22) & (z_img*1e3 <= 38))
zext = [x_img[zx][0]*1e3, x_img[zx][-1]*1e3, z_img[zz][-1]*1e3, z_img[zz][0]*1e3]
row = np.argmin(np.abs(z_img - 0.020))

fig = plt.figure(figsize=(14, 13))
gs = fig.add_gridspec(3, 4, height_ratios=[1.25, 1, 0.55])
titles = ["before filter", "after filter (used: eps 0.001)", "stronger filter (eps 0.01, NOT used)"]
for j, (im, t) in enumerate(zip((image, chosen, strong), titles)):
    a = fig.add_subplot(gs[0, j]); a.imshow(im, cmap="gray", vmin=0, vmax=1, extent=ext)
    a.add_patch(plt.Rectangle((zext[0], zext[3]), zext[1]-zext[0], zext[2]-zext[3], fill=False, ec="yellow", lw=1.5))
    a.set(title=t, xlabel="x (mm)", ylabel="z (mm)")
    b = fig.add_subplot(gs[1, j]); b.imshow(im[np.ix_(zz, zx)], cmap="gray", vmin=0, vmax=1, extent=zext, interpolation="nearest")
    b.set(title="zoom (yellow box)", xlabel="x (mm)", ylabel="z (mm)")
a = fig.add_subplot(gs[0, 3]); d = np.abs(chosen - image)
im_ = a.imshow(np.clip(10*d, 0, 1), cmap="magma", vmin=0, vmax=1, extent=ext)
a.set(title="|after - before| x10\n(where the filter acted)", xlabel="x (mm)"); fig.colorbar(im_, ax=a, fraction=0.05)
b = fig.add_subplot(gs[1, 3]); b.imshow(np.clip(10*d[np.ix_(zz, zx)], 0, 1), cmap="magma", vmin=0, vmax=1, extent=zext, interpolation="nearest")
b.set(title="difference, zoom", xlabel="x (mm)")
c = fig.add_subplot(gs[2, :])
for im, lab, st in ((image, "before", "-"), (chosen, "after (eps 0.001)", "-"), (strong, "eps 0.01 (not used)", ":")):
    c.plot(x_img*1e3, im[row], st, lw=1.3, label=lab)
c.set(title=f"Brightness along one row (z = {z_img[row]*1e3:.1f} mm)", xlabel="x (mm)", ylabel="brightness"); c.legend(ncol=3)
plt.tight_layout(); plt.show()

u0 = to_uint8(image)
for nm, im in (("after, eps 0.001 (used)", chosen), ("eps 0.01 (not used)", strong)):
    print(f"{nm:24s}: speckle index {speckle_index(u0):.3f} -> {speckle_index(to_uint8(im)):.3f}")

image, x_img, z_img = _keep

**How to read the plots:** in the zooms, the "after" image has softer, less grainy speckle while the cyst border stays crisp. The ε 0.01 image is visibly blurrier at the border, which is why it was rejected. The difference map is brightest in the speckle and weakest inside the black cyst (nothing to smooth there).

---
# FINAL IMAGE

The output of the whole chain: **T1 → T2 → T3 → T4 → T5 → T6 → T7**. Brightness 1 = strongest echo (0 dB), 0 = 50 dB weaker or less. x = across the probe, z = depth into the body.

In [ ]:
plt.figure(figsize=(4, 8))
plt.imshow(final_bmode, cmap="gray", vmin=0, vmax=1, extent=extent)
plt.colorbar(label=f"brightness (0 = -{config.DYNAMIC_RANGE_DB:g} dB, 1 = 0 dB)", fraction=0.08)
plt.xlabel("lateral x (mm)"); plt.ylabel("depth z (mm)"); plt.title(f"Final B-mode: {KIND}"); plt.show()

**What this image shows (`cyst_lesion`):** speckled soft tissue with a dark round **cyst** (6 mm radius, at 30 mm depth).

Quality numbers from the full run of `simulator/main.py`, same settings:

| Measure | Value | Meaning |
|---|---|---|
| Cyst contrast | **−19.4 dB** vs surrounding tissue | the cyst is clearly darker |
| CNR (before → after filter) | 2.323 → **2.425** | contrast-to-noise; higher = cyst easier to see |
| gCNR (before → after filter) | 0.860 → **0.871** | 0 = cyst indistinguishable, 1 = perfectly separable |
| Speckle SNR (`scatterers`) | 1.73-1.92 | theory for fully developed speckle: 1.91 |
| Point targets (`point_targets`) | within 0.04 mm of their true positions | geometry is correct |
| Point width, axial / lateral | 0.33-0.35 / 0.47 mm | resolution |

### Handover summary: what each team produced in this run
The table printed below is the "contract" between the teams: each output's name, shape and units. Each team's output shape must match what the next team expects.

In [ ]:
rows = [("T1", "element_positions", element_positions, "m"),
        ("T2", "pulse", pulse, "a.u."), ("T2", "tx_delays", tx_delays, "s"), ("T2", "transmit_field", field_b, "a.u. (grid)"),
        ("T3", "propagated_wavefield", propagated_wavefield, "a.u. vs t_axis"), ("T3", "travel_times", travel_times, "s"),
        ("T4", "raw_rf", raw_rf, "a.u., elements x samples"), ("T4", "t_axis", t_axis, "s"),
        ("T5", "beamformed_rf", beamformed_rf, "a.u., depth x lines"), ("T5", "z_axis", z_axis, "m"),
        ("T6", "bmode", bmode, "0..1 (50 dB)"), ("T7", "image", image, "0..1, px"), ("T7", "final_bmode", final_bmode, "0..1, px")]
print(f"{'team':5s} {'output':22s} {'shape':14s} units")
for team, name, arr, unit in rows:
    print(f"{team:5s} {name:22s} {str(np.shape(arr)):14s} {unit}")

---
## Bonus: all three required phantoms

Runs the complete chain (T4 → T5 → T6 → T7) for each phantom, side by side. Measured on the 8-core Mac: point targets 3 s, scatterers 257 s, cyst 255 s (≈8.5 min in total).

In [ ]:
def make_image(kind):
    rf, t = simulate_rf(build_phantom(kind))
    bf, z = das_beamform(rf, EX, x_lines, config, t_axis=t)
    img, xi, zi = scan_convert(bmode_formation(bf), x_lines, z)
    return postprocess(img), xi, zi

fig, axes = plt.subplots(1, 3, figsize=(12, 8))
for a, kind in zip(axes, ["point_targets", "scatterers", "cyst_lesion"]):
    t0 = time.time(); img, xi, zi = make_image(kind)
    print(f"{kind}: {time.time()-t0:.1f} s")
    a.imshow(img, cmap="gray", vmin=0, vmax=1, extent=[xi[0]*1e3, xi[-1]*1e3, zi[-1]*1e3, zi[0]*1e3])
    a.set(title=kind, xlabel="x (mm)", ylabel="z (mm)")
plt.suptitle(f"NITK-UsoundSim final images ({config.DYNAMIC_RANGE_DB:g} dB)"); plt.tight_layout(); plt.show()

**How to read the three images:**
- **point_targets:** three small bright dots at (−5, 10), (0, 20), (+5, 30) mm. How small the dots are shows the resolution. A faint diagonal line near the shallow target (≈ −25 dB) is a known simulation artefact; its cause has not been found.
- **scatterers:** uniform speckle, the texture of homogeneous soft tissue, fading with depth (attenuation, no TGC).
- **cyst_lesion:** the same speckle with a black circle: an anechoic (fluid-filled) cyst.

## Known limitations
- **Real body model:** there is none. The phantoms are standard test objects, not an organ.
- **Pitch and f0:** 0.30 mm and 8 MHz are T2's assumptions; Philips does not publish them.
- **Depth darkening:** the image is ≈ −17 dB darker at 36-40 mm, because there is no TGC (time-gain compensation).
- **Diagonal artefact:** the −25 dB line near the (−5, 10) mm target is not root-caused.

## More documentation
- `docs/README.md`: project history and verified results.
- `docs/MODULES.md`: every module's inputs, outputs and equations.
- `docs/NITK-UsoundSim_Project_Flow.pdf`: a one-page flow of which output feeds which input.